# Local LLM Framework Showdown: vLLM vs. llama.cpp vs. Ollama

**NAIRR Workshop 2026 — running the *same* model three different ways on a Jetstream2 GPU instance**

We run **one model — Qwen3-4B — across three popular local-inference engines** and benchmark them head-to-head on the same hardware.

| Engine | What it is | Strength | Model format |
|--------|-----------|----------|--------------|
| **Ollama** | Easiest "it just works" server. One command to pull & run. | Simplicity, great defaults | GGUF (managed for you) |
| **llama.cpp** | The C/C++ engine under Ollama. Maximum portability, runs on CPU or GPU. | Lightweight, quantized, runs anywhere | GGUF |
| **vLLM** | Production serving engine. PagedAttention + continuous batching. | Throughput & concurrency at scale | HuggingFace (fp16/bf16) |

> **This notebook adapts to your instance.** On a **CPU instance** (e.g. Jetstream2 `m3.quad`) it compares **Ollama vs llama.cpp** — both are built to run fast on CPU — and **skips vLLM**, which is GPU-first. On a **GPU instance** (`g3` / `g4` / `g5`) it automatically adds **vLLM** for the full three-way showdown. The cell in Section 2 detects which you're on.

### The key idea to teach
> **Same weights → the framework mainly changes SPEED. Quantization mainly changes QUALITY.**
>
> Ollama and llama.cpp run a **4-bit quantized GGUF**; vLLM runs the **16-bit HuggingFace** weights. So any *quality* gap you see is mostly **quantization**, and any *speed* gap is mostly the **engine**. We measure both.

### What we measure (full benchmark)
- **Latency** — total wall-clock per request
- **TTFT** — time to first token (responsiveness)
- **Throughput** — tokens / second during generation
- **Memory** — peak GPU VRAM used
- **Load time** — how long the engine takes to load the model
- **Quality** — side-by-side answers so you can judge for yourself
- **Batch throughput** — all prompts at once (this is where vLLM pulls ahead)

## 1. Configuration

Everything you might want to change lives here. **The model is one variable** — swap `qwen3:4b` / `Qwen/Qwen3-4B` for any size you like (e.g. `qwen3:1.7b` for a faster demo, `qwen3:8b` for stronger answers).

Use **`TEST_MODE`** to choose how much to run: **`"single"`** (one prompt — fastest, for a very short class), **`"simple"`** (3 prompts — a good live demo), or **`"complete"`** (all prompts — the full benchmark later at home). Progress bars show how far along each engine is.

In [ ]:
# ============================ CONFIG ============================
# Model identity across the three engines (must be the SAME model!)
MODEL_OLLAMA = "qwen3:4b"            # Ollama tag       -> GGUF, 4-bit
MODEL_HF     = "Qwen/Qwen3-4B"       # HuggingFace repo -> vLLM, 16-bit
GGUF_REPO    = "Qwen/Qwen3-4B-GGUF"  # GGUF repo        -> llama.cpp
GGUF_QUANT   = "Q4_K_M"              # quant to pick from the GGUF repo

# Qwen3 is a hybrid reasoning model. Thinking inflates token counts and
# makes a SPEED comparison unfair, so we disable it by default.
# Flip to True to watch the model "think" (much slower, more tokens).
ENABLE_THINKING = False

# ---- Test depth: how much to run ----
# "single"   = ONE prompt          -> fastest possible demo (very short class)
# "simple"   = 3 prompts, short     -> good live demo
# "complete" = all 7 prompts, long  -> run later at home for the full picture
TEST_MODE = "simple"

if TEST_MODE == "single":
    PROMPT_SUBSET = ["sky"]                          # one clear, quick prompt
    MAX_TOKENS    = 256
elif TEST_MODE == "simple":
    PROMPT_SUBSET = ["greeting", "sky", "primes"]    # fast + illustrative
    MAX_TOKENS    = 256
else:                                                # "complete"
    PROMPT_SUBSET = None                             # use every prompt
    MAX_TOKENS    = 512

# Generation settings (identical for every engine -> fair comparison)
TEMPERATURE = 0.7
TOP_P       = 0.9

# Stream each model's answer to the screen as it generates (great for a live demo).
SHOW_OUTPUT = True

# vLLM control: None = auto (run only if a GPU is found). True/False to force.
FORCE_VLLM  = None

# Set False on a re-run to skip the (slow) dependency install
RUN_INSTALL = True
# ===============================================================
print("Config loaded. Model:", MODEL_HF, "| thinking:", ENABLE_THINKING)
print("Test mode:", TEST_MODE, "| max tokens per answer:", MAX_TOKENS)
# ---------------------------------------------------------------------------
# CPU DEMO TIP (m3.quad = 4 vCPU, no GPU):
#   Qwen3-4B in 4-bit runs on CPU but the long 'essay' prompt can take a while.
#   For a snappier live demo, set MODEL_OLLAMA="qwen3:1.7b",
#   MODEL_HF="Qwen/Qwen3-1.7B", GGUF_REPO="Qwen/Qwen3-1.7B-GGUF" and MAX_TOKENS=256.
# ---------------------------------------------------------------------------

## 2. Environment check

Confirm we're on a Linux box with an NVIDIA GPU. vLLM **requires** this; llama.cpp and Ollama will happily use it too.

In [ ]:
import subprocess, platform, sys, os

print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())
print("CPUs   :", os.cpu_count())

HAS_GPU = False
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True, check=True).stdout.strip()
    print("GPU    :", out)
    HAS_GPU = True
except Exception:
    print("GPU    : none detected -> CPU-only instance (e.g. m3.quad)")

# Derived runtime flags used throughout the notebook
RUN_VLLM     = HAS_GPU if FORCE_VLLM is None else FORCE_VLLM
N_GPU_LAYERS = -1 if HAS_GPU else 0          # llama.cpp: all-on-GPU vs CPU

print("-" * 60)
if RUN_VLLM:
    print(">> GPU mode: comparing Ollama + llama.cpp + vLLM (3 engines)")
else:
    print(">> CPU mode: comparing Ollama + llama.cpp (vLLM skipped - needs a GPU)")
    print("   Spin up a g3/g4/g5 GPU flavor to add vLLM.")

## 3. Install the three engines

This is the slow cell (a few minutes — vLLM pulls a lot). Run once.

- **vLLM** via pip (pins its own torch build)
- **llama-cpp-python** from the prebuilt **CUDA** wheel index (no compiling)
- **Ollama** via the official install script

> If the llama-cpp CUDA wheel URL doesn't match your driver, change `cu124` to your CUDA version (e.g. `cu121`, `cu122`, `cu125`).

In [ ]:
if RUN_INSTALL:
    import sys, subprocess
    def pip(*args):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

    print("[1/4] utilities ...");  pip("requests", "pandas", "matplotlib", "huggingface_hub", "tabulate", "tqdm", "ipywidgets")

    print("[2/4] llama-cpp-python ...")
    if HAS_GPU:   # prebuilt CUDA wheel (no compiling)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python",
                        "--extra-index-url",
                        "https://abetlen.github.io/llama-cpp-python/whl/cu124"], check=True)
    else:         # prebuilt CPU wheel
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python",
                        "--extra-index-url",
                        "https://abetlen.github.io/llama-cpp-python/whl/cpu"], check=True)

    print("[3/4] vllm ...", "(skipped - no GPU)" if not RUN_VLLM else "")
    if RUN_VLLM:
        pip("vllm")

    print("[4/4] ollama ...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    print("Done.")
else:
    print("Skipped install (RUN_INSTALL=False)")

## 4. The benchmark prompts — *designed to expose real differences*

You asked for better questions than just "hi" and "write a prime sieve". The trick: each prompt **stresses a different dimension**, because the same weights in three engines will give *similar quality* — so the interesting differences are about **speed under different workloads** and **quality under quantization**.

| # | Prompt | What it stresses | Why it reveals a difference |
|---|--------|------------------|------------------------------|
| `greeting` | "Hi, who are you?" | **Latency floor / TTFT** | Tiny output → you're measuring pure engine overhead & responsiveness |
| `sky` | "Why is the sky blue?" | **Short explanation** | Classic baseline; quick to eyeball quality |
| `primes` | "Write an efficient Python script for primes 1–10,000" | **Structured code generation** | Code must be correct → quantization (4-bit GGUF) can introduce subtle bugs the fp16 vLLM run won't |
| `essay` | "Write a ~400-word explanation of photosynthesis" | **Sustained throughput** | Long generation → the *cleanest* tokens/sec signal |
| `reasoning` | Multi-step discount word problem | **Reasoning under quantization** | The dimension most likely to show a *quality* gap between 4-bit and 16-bit |
| `json` | "Return the 8 planets as strict JSON" | **Format adherence** | Instruction-following / valid JSON can degrade with quantization |
| `longctx` | Long passage + a question about it | **Prompt-processing (prefill) speed** | Big input → measures how fast each engine ingests context, where engines differ most |

We then finish with a **batch test** (all prompts at once) — the workload where vLLM's continuous batching dominates.

In [ ]:
_PASSAGE = (
    "The National Artificial Intelligence Research Resource (NAIRR) is a shared "
    "national infrastructure that provides researchers and educators access to "
    "computing, data, models, and training. A pilot launched in 2024 to connect "
    "the U.S. research community to resources such as Jetstream2, a cloud "
    "computing system. Jetstream2 offers GPU-backed virtual machines that can be "
    "shared across a class through a single allocation, with each student using "
    "their own ACCESS identity. The goal is to lower the barrier to AI research "
    "for institutions that lack large on-premise GPU clusters, while keeping "
    "usage accountable through per-user credit tracking."
)

PROMPTS = {
    "greeting":  "Hi! In one short sentence, who are you?",
    "sky":       "Why is the sky blue? Explain in 3-4 sentences.",
    "primes":    ("Write an efficient Python script that finds all prime numbers "
                  "from 1 to 10,000 using the Sieve of Eratosthenes, and prints "
                  "how many primes it found. Only output the code."),
    "essay":     ("Write a clear, detailed ~400-word explanation of how "
                  "photosynthesis works, written for a high-school student."),
    "reasoning": ("A store offers 25% off an item, then takes an additional 10% "
                  "off the already-discounted price. What is the single overall "
                  "percentage discount off the original price? Show your reasoning "
                  "step by step, then give the final number."),
    "json":      ('Return ONLY a JSON array of the 8 planets of our solar system. '
                  'Each element must be an object with keys "name", '
                  '"diameter_km" (number), and "moons" (number). No prose.'),
    "longctx":   ("Read the passage below and answer in one sentence: according to "
                  "the text, how does Jetstream2 keep shared classroom usage "
                  f"accountable?\n\nPASSAGE:\n{_PASSAGE}"),
}
if PROMPT_SUBSET:                       # TEST_MODE == "simple"
    PROMPTS = {k: PROMPTS[k] for k in PROMPT_SUBSET if k in PROMPTS}
print(f"[{TEST_MODE} mode] {len(PROMPTS)} prompts ready:", ", ".join(PROMPTS))
print("Tip: set TEST_MODE='complete' in the config cell for the full run.")

## 5. Shared helpers — metrics, VRAM, prompt formatting

Every engine call returns the **same standardized record** so the final table is apples-to-apples. We also print **system RAM** at each model load/unload — watch these numbers rise and fall, and compare them against `btop`/`htop` running in a side terminal.

In [ ]:
import time, subprocess, gc, json
from tqdm.auto import tqdm   # progress bars

RESULTS = []   # every benchmark row lands here

def gpu_mem_used_mib():
    """Current GPU memory in use (MiB) via nvidia-smi. Returns -1 if unavailable."""
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True).stdout.strip().splitlines()[0]
        return int(out)
    except Exception:
        return -1

def record(framework, name, prompt_toks, completion_toks,
           ttft_s, total_s, vram_mib, output):
    gen_s = max(total_s - (ttft_s or 0), 1e-6)
    tps   = completion_toks / gen_s if completion_toks else 0.0
    row = {
        "framework": framework, "prompt": name,
        "prompt_tokens": prompt_toks, "completion_tokens": completion_toks,
        "ttft_s": round(ttft_s, 3) if ttft_s is not None else None,
        "total_s": round(total_s, 3),
        "tokens_per_sec": round(tps, 1),
        "vram_mib": vram_mib,
        "output": output,
    }
    RESULTS.append(row)
    short = output.replace("\n", " ")[:70]
    tqdm.write(f"  [{framework:9}] {name:10} {completion_toks:4d} tok  "
               f"{total_s:6.2f}s  {tps:6.1f} tok/s  | {short}")
    return row

# Qwen3 chat messages. /no_think keeps the speed test fair when thinking is off.
def build_messages(prompt):
    user = prompt if ENABLE_THINKING else prompt + " /no_think"
    return [{"role": "user", "content": user}]

def mem_report(label=""):
    """Print system RAM used/free (MB) from /proc/meminfo. Match against btop."""
    try:
        info = {}
        with open("/proc/meminfo") as f:
            for line in f:
                key, val = line.split(":", 1)
                info[key] = int(val.split()[0])          # values are in kB
        total = info["MemTotal"] / 1024
        avail = info.get("MemAvailable", info["MemFree"]) / 1024
        used = total - avail
        print(f"   [RAM] {label:26} used {used:6.0f} MB | free {avail:6.0f} MB "
              f"({used/total*100:.0f}% of {total:.0f} MB)")
    except Exception as e:
        print("   [RAM] unavailable:", e)

print("Helpers ready.")
mem_report("baseline (no model loaded)")

## 6. Engine 1 — Ollama  🦙

The "it just works" path. We start the server, pull the model, and hit its HTTP API.
Ollama's response conveniently **reports its own token counts and timings**, so the stats here come straight from the engine.

In [ ]:
import requests, time, subprocess, os

# start the server in the background
os.environ.setdefault("OLLAMA_HOST", "127.0.0.1:11434")
subprocess.Popen(["ollama", "serve"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# wait until it's up
OLLAMA_URL = "http://127.0.0.1:11434"
for _ in range(60):
    try:
        if requests.get(OLLAMA_URL + "/api/tags", timeout=2).ok:
            print("Ollama server is up."); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start")

print(f"Pulling {MODEL_OLLAMA} (first time downloads weights) ...")
subprocess.run(["ollama", "pull", MODEL_OLLAMA], check=True)
print("Model ready.")

In [ ]:
def run_ollama(name, prompt):
    body = {
        "model": MODEL_OLLAMA,
        "messages": build_messages(prompt),
        "stream": True,                     # stream so we can show the answer live
        "think": ENABLE_THINKING,
        "options": {"temperature": TEMPERATURE, "top_p": TOP_P,
                    "num_predict": MAX_TOKENS},
    }
    if SHOW_OUTPUT:
        print(f"\n--- [ollama] {name} ---")
    t0 = time.time(); first = None; ttft = None; text = ""; p_tok = c_tok = 0
    with requests.post(OLLAMA_URL + "/api/chat", json=body, stream=True, timeout=600) as r:
        for line in r.iter_lines():
            if not line:
                continue
            obj = json.loads(line)
            piece = obj.get("message", {}).get("content", "")
            if piece:
                if first is None:
                    first = time.time(); ttft = first - t0   # real time-to-first-token
                text += piece
                if SHOW_OUTPUT:
                    print(piece, end="", flush=True)
            if obj.get("done"):
                p_tok = obj.get("prompt_eval_count", 0)
                c_tok = obj.get("eval_count", 0)
    total_s = time.time() - t0
    if SHOW_OUTPUT:
        print()
    vram = gpu_mem_used_mib()
    return record("ollama", name, p_tok, c_tok, ttft, total_s, vram, text)

print("Running Ollama benchmark:")
OLLAMA_LOAD_T0 = time.time()
for nm, pr in tqdm(PROMPTS.items(), total=len(PROMPTS), desc="Ollama"):
    run_ollama(nm, pr)
mem_report("after Ollama benchmark")   # model is loaded in RAM here

In [ ]:
# Free Ollama's memory before the next engine.
# IMPORTANT: use /api/generate with keep_alive=0 and NO prompt -> this unloads the
# model WITHOUT generating anything. (Sending a chat message instead would make
# Qwen3 "think" with no token limit and hang for minutes on CPU.)
mem_report("before Ollama unload")
requests.post(OLLAMA_URL + "/api/generate",
              json={"model": MODEL_OLLAMA, "keep_alive": 0}, timeout=30)
for _ in range(40):                          # poll until the model is actually gone
    if not requests.get(OLLAMA_URL + "/api/ps", timeout=5).json().get("models", []):
        break
    time.sleep(0.5)
print("Ollama model unloaded. VRAM now:", gpu_mem_used_mib(), "MiB")
mem_report("after Ollama unload")   # watch 'free' (available) rise here

## 7. Engine 2 — llama.cpp (via `llama-cpp-python`)  ⚙️

The lightweight C/C++ engine that powers Ollama under the hood. We load the **same GGUF** directly and offload all layers to the GPU (`n_gpu_layers=-1`). We stream so we can measure **TTFT** precisely.

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download

# auto-pick the right quant file from the GGUF repo (robust to naming)
files = [f for f in list_repo_files(GGUF_REPO) if f.endswith(".gguf")]
match = [f for f in files if GGUF_QUANT.lower() in f.lower()]
gguf_name = (match or files)[0]
print("Selected GGUF:", gguf_name, "(from", len(files), "files)")
gguf_path = hf_hub_download(repo_id=GGUF_REPO, filename=gguf_name)
print("Downloaded to:", gguf_path)

In [ ]:
from llama_cpp import Llama

t0 = time.time()
llm_cpp = Llama(
    model_path=gguf_path,
    n_gpu_layers=N_GPU_LAYERS,   # -1 = all on GPU, 0 = pure CPU
    n_ctx=4096,
    n_threads=os.cpu_count(),
    verbose=False,
)
LLAMACPP_LOAD_S = time.time() - t0
print(f"llama.cpp loaded in {LLAMACPP_LOAD_S:.1f}s "
      f"({'GPU' if N_GPU_LAYERS else 'CPU'}) | VRAM {gpu_mem_used_mib()} MiB")
mem_report("after llama.cpp load")

In [ ]:
def run_llamacpp(name, prompt):
    msgs = build_messages(prompt)
    t0 = time.time(); first = None; text = ""
    stream = llm_cpp.create_chat_completion(
        messages=msgs, max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE, top_p=TOP_P, stream=True)
    if SHOW_OUTPUT:
        print(f"\n--- [llama.cpp] {name} ---")
    for chunk in stream:
        delta = chunk["choices"][0]["delta"].get("content", "")
        if delta:
            if first is None:
                first = time.time()
            text += delta
            if SHOW_OUTPUT:
                print(delta, end="", flush=True)
    if SHOW_OUTPUT:
        print()
    total_s = time.time() - t0
    ttft = (first - t0) if first else None
    vram = gpu_mem_used_mib()
    p_tok = len(llm_cpp.tokenize(prompt.encode()))
    c_tok = len(llm_cpp.tokenize(text.encode())) if text else 0
    return record("llama.cpp", name, p_tok, c_tok, ttft, total_s, vram, text)

print("Running llama.cpp benchmark:")
for nm, pr in tqdm(PROMPTS.items(), total=len(PROMPTS), desc="llama.cpp"):
    run_llamacpp(nm, pr)

In [ ]:
# free llama.cpp memory before vLLM
mem_report("before llama.cpp unload")
del llm_cpp
gc.collect()
time.sleep(2)
print("llama.cpp unloaded. VRAM now:", gpu_mem_used_mib(), "MiB")
mem_report("after llama.cpp unload")

## 8. Engine 3 — vLLM  🚀  *(GPU only — auto-skipped on CPU instances)*

The production engine. It loads the **16-bit HuggingFace** weights (not quantized), so expect **higher VRAM** but **top-tier throughput** — especially in the batch test later. We use vLLM's offline `LLM` API and read its built-in per-request metrics for TTFT.

**On `m3.quad` (CPU) these cells will print a skip message and do nothing** — the comparison continues with Ollama vs llama.cpp.

In [ ]:
VLLM_LOAD_S = None
if not RUN_VLLM:
    print("Skipping vLLM (no GPU). Sections 9-11 will compare Ollama vs llama.cpp.")
else:
    from vllm import LLM, SamplingParams
    from transformers import AutoTokenizer

    t0 = time.time()
    llm_vllm = LLM(model=MODEL_HF, dtype="bfloat16",
                   gpu_memory_utilization=0.85, max_model_len=4096,
                   enforce_eager=True)
    VLLM_LOAD_S = time.time() - t0
    tok = AutoTokenizer.from_pretrained(MODEL_HF)
    print(f"vLLM loaded in {VLLM_LOAD_S:.1f}s | VRAM {gpu_mem_used_mib()} MiB")
    mem_report("after vLLM load")

In [ ]:
if RUN_VLLM:
    def _format(prompt):
        return tok.apply_chat_template(
            build_messages(prompt), tokenize=False, add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING)

    sp = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS)

    def run_vllm(name, prompt):
        out = llm_vllm.generate([_format(prompt)], sp)[0]
        o = out.outputs[0]
        p_tok = len(out.prompt_token_ids)
        c_tok = len(o.token_ids)
        m = out.metrics
        ttft = (m.first_token_time - m.arrival_time) if m and m.first_token_time else None
        total_s = (m.finished_time - m.arrival_time) if m else 0.0
        vram = gpu_mem_used_mib()
        if SHOW_OUTPUT:                       # vLLM (offline) can't stream; show full text
            print(f"\n--- [vllm] {name} ---\n{o.text}")
        return record("vllm", name, p_tok, c_tok, ttft, total_s, vram, o.text)

    print("Running vLLM benchmark:")
    for nm, pr in tqdm(PROMPTS.items(), total=len(PROMPTS), desc="vLLM"):
        run_vllm(nm, pr)
else:
    print("vLLM skipped (CPU instance).")

## 9. The comparison table 📊

Same model, same prompts, same GPU — three engines. Now we line them up.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 50)

df = pd.DataFrame(RESULTS)

print("=== Average performance per engine ===")
summary = (df.groupby("framework")
             .agg(avg_tokens_per_sec=("tokens_per_sec", "mean"),
                  avg_ttft_s=("ttft_s", "mean"),
                  avg_total_s=("total_s", "mean"),
                  peak_vram_mib=("vram_mib", "max"),
                  total_completion_tokens=("completion_tokens", "sum"))
             .round(2).sort_values("avg_tokens_per_sec", ascending=False))
_load = {"ollama": None, "llama.cpp": round(LLAMACPP_LOAD_S, 1)}
if VLLM_LOAD_S is not None:
    _load["vllm"] = round(VLLM_LOAD_S, 1)
summary["load_time_s"] = pd.Series(_load)
summary

In [ ]:
# tokens/sec for every prompt x engine (the headline grid)
print("=== Throughput (tokens/sec) by prompt ===")
pivot_tps = df.pivot(index="prompt", columns="framework", values="tokens_per_sec")
pivot_tps.loc["** AVG **"] = pivot_tps.mean()
pivot_tps.round(1)

In [ ]:
# latency grid
print("=== Total latency (seconds) by prompt ===")
df.pivot(index="prompt", columns="framework", values="total_s").round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
summary["avg_tokens_per_sec"].plot.bar(ax=ax[0], color=["#4c9", "#49c", "#c49"])
ax[0].set_title("Avg throughput (higher = better)"); ax[0].set_ylabel("tokens / sec")
ax[0].tick_params(axis="x", rotation=0)

summary["avg_ttft_s"].plot.bar(ax=ax[1], color=["#4c9", "#49c", "#c49"])
ax[1].set_title("Avg time-to-first-token (lower = better)"); ax[1].set_ylabel("seconds")
ax[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

## 10. Quality side-by-side 🔍

Remember: Ollama & llama.cpp run **4-bit** weights, vLLM runs **16-bit**. Read the answers and look for places where the quantized engines drift — most likely on `reasoning`, `primes` (a subtle bug), or `json` (format slips).

In [ ]:
def show_prompt(name, limit=600):
    print("=" * 90)
    print("PROMPT:", PROMPTS[name][:120], "...\n")
    for fw in ["ollama", "llama.cpp", "vllm"]:
        rows = [r for r in RESULTS if r["prompt"] == name and r["framework"] == fw]
        if not rows: continue
        r = rows[0]
        print(f"--- {fw}  ({r['completion_tokens']} tok, {r['tokens_per_sec']} tok/s) ---")
        print(r["output"][:limit].strip())
        print()

# change the key to inspect any prompt: greeting sky primes essay reasoning json longctx
show_prompt("reasoning")

In [ ]:
show_prompt("primes")

## 11. The batch test — where vLLM earns its reputation 🏎️

So far we sent **one prompt at a time** — that hides vLLM's superpower. Real serving means **many concurrent requests**. vLLM's continuous batching processes them together; Ollama/llama.cpp largely handle them one after another.

We send **all prompts at once** and compare aggregate throughput. *(On CPU, the vLLM row is skipped — but the Ollama concurrency test still runs.)*

In [ ]:
vllm_batch_s = vllm_batch_tokens = None
if RUN_VLLM:
    # vLLM: true batch -- one generate() call with all prompts
    batch_prompts = [_format(p) for p in PROMPTS.values()]
    t0 = time.time()
    outs = llm_vllm.generate(batch_prompts, sp)
    vllm_batch_s = time.time() - t0
    vllm_batch_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
    print(f"vLLM   batch: {len(batch_prompts)} prompts, {vllm_batch_tokens} tokens "
          f"in {vllm_batch_s:.2f}s  ->  {vllm_batch_tokens/vllm_batch_s:.1f} tok/s aggregate")
else:
    print("vLLM batch skipped (CPU instance).")

In [ ]:
# Ollama: concurrent requests via threads (closest fair comparison)
from concurrent.futures import ThreadPoolExecutor

def _ollama_once(prompt):
    body = {"model": MODEL_OLLAMA, "messages": build_messages(prompt),
            "stream": False, "think": ENABLE_THINKING,
            "options": {"temperature": TEMPERATURE, "num_predict": MAX_TOKENS}}
    r = requests.post(OLLAMA_URL + "/api/chat", json=body, timeout=600).json()
    return r.get("eval_count", 0)

# reload ollama model first
requests.post(OLLAMA_URL + "/api/chat",
              json={"model": MODEL_OLLAMA, "messages":[{"role":"user","content":"hi"}],
                    "stream": False})
t0 = time.time()
with ThreadPoolExecutor(max_workers=len(PROMPTS)) as ex:
    counts = list(ex.map(_ollama_once, PROMPTS.values()))
ollama_batch_s = time.time() - t0
ollama_batch_tokens = sum(counts)
print(f"Ollama batch: {len(PROMPTS)} prompts, {ollama_batch_tokens} tokens "
      f"in {ollama_batch_s:.2f}s  ->  {ollama_batch_tokens/ollama_batch_s:.1f} tok/s aggregate")

In [ ]:
rows = [{"engine": "Ollama (threaded)",
         "prompts": len(PROMPTS), "tokens": ollama_batch_tokens,
         "wall_s": round(ollama_batch_s, 2),
         "aggregate_tok_per_s": round(ollama_batch_tokens / ollama_batch_s, 1)}]
if vllm_batch_s:
    rows.insert(0, {"engine": "vLLM (continuous batching)",
                    "prompts": len(PROMPTS), "tokens": vllm_batch_tokens,
                    "wall_s": round(vllm_batch_s, 2),
                    "aggregate_tok_per_s": round(vllm_batch_tokens / vllm_batch_s, 1)})
print("=== Batch / concurrency throughput ===")
pd.DataFrame(rows)

## 12. How to read these results 🎓

**Single-request speed (Sections 9–10)**
- **llama.cpp & Ollama** (4-bit GGUF) are usually **fastest for one request** and use the **least VRAM** — the quantized model is smaller, so each token is cheaper. Great for a laptop, a single user, or a workshop demo.
- **vLLM** (16-bit) may look slower on *one* request and uses **more VRAM**, because it's optimized for *many* requests, not one.

**Batch / concurrency (Section 11)**
- **vLLM pulls dramatically ahead** when prompts arrive together. Continuous batching + PagedAttention keep the GPU saturated. This is why production APIs run vLLM, not Ollama.

**Quality (Section 10)**
- Answers are *mostly* similar because **it's the same model**. Where they differ, it's usually the **4-bit quantization** (Ollama/llama.cpp) vs **16-bit** (vLLM) — watch `reasoning`, `primes`, and `json`.

**Pick-the-engine cheat sheet**

| You want... | Use |
|-------------|-----|
| Easiest setup, single user, prototyping | **Ollama** |
| Embedded / max portability / squeeze onto small GPU | **llama.cpp** |
| Serve many users, highest throughput, full precision | **vLLM** |

### Try next
- Set `ENABLE_THINKING = True` and re-run — watch token counts explode and speed drop.
- Swap `MODEL_OLLAMA="qwen3:8b"` + `MODEL_HF="Qwen/Qwen3-8B"` to see the size/speed/quality trade-off.
- Add your own prompt to `PROMPTS` and see which engine handles it best.